In [ ]:
# CELL 2: Download dataset langsung dari Kaggle (Wildan Sani)
import os
import zipfile
import json

print(" DOWNLOAD DATASET WASTE CLASSIFICATION")

# Langkah 1: Setup Kaggle API dengan akun wildansani
print("\n Konfigurasi Kaggle API...")

!mkdir -p ~/.kaggle

# Konfigurasi dengan username dan key kamu
kaggle_json = {
    "username": "wildansani",
    "key": "fc1607894987c6602018c570db0106b0"
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_json, f)

!chmod 600 ~/.kaggle/kaggle.json
print(" Kaggle API siap!")

# Langkah 2: Install Kaggle API
print("\n Install Kaggle API...")
!pip install kaggle -q
print(" Kaggle API terinstall!")

# Langkah 3: Download dataset
print("\n Download dataset...")
print("   Sumber: techsash/waste-classification-data")
print("   Ukuran: ~225 MB (25.000+ gambar)")

result = !kaggle datasets download techsash/waste-classification-data --force

print(" Download selesai!")

# Langkah 4: Ekstrak
print("\n Ekstrak dataset...")

# Cari file zip
zip_file = None
for f in os.listdir():
    if f.endswith('.zip'):
        zip_file = f
        break

if zip_file:
    print(f"   File: {zip_file}")

    # Buat folder data
    os.makedirs('/content/data', exist_ok=True)

    # Ekstrak
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall('/content/data')

    print(" Ekstrak selesai!")

    # Hapus zip
    os.remove(zip_file)
    print(" File ZIP dihapus (hemat space)")
else:
    print(" File ZIP tidak ditemukan!")
    exit()

# Langkah 5: Cari TRAIN dan TEST path
print("\n Mencari folder TRAIN dan TEST...")

data_dir = '/content/data'
train_path = None
test_path = None

# Cek struktur
if os.path.exists('/content/data/dataset'):
    dataset_dir = '/content/data/dataset'
    for item in os.listdir(dataset_dir):
        if item.upper() == 'TRAIN':
            train_path = os.path.join(dataset_dir, item)
        elif item.upper() == 'TEST':
            test_path = os.path.join(dataset_dir, item)

# Jika tidak ditemukan, coba cari di root
if not train_path:
    for root, dirs, files in os.walk(data_dir):
        if 'TRAIN' in dirs:
            train_path = os.path.join(root, 'TRAIN')
        if 'TEST' in dirs:
            test_path = os.path.join(root, 'TEST')

# Tampilkan hasil
if train_path and test_path:
    print(f" TRAIN: {train_path}")
    print(f" TEST:  {test_path}")

    # Statistik
    print("\n JUMLAH GAMBAR:")
    for cat in ['O', 'R']:
        train_cat = os.path.join(train_path, cat)
        test_cat = os.path.join(test_path, cat)
        if os.path.exists(train_cat):
            train_count = len(os.listdir(train_cat))
            test_count = len(os.listdir(test_cat)) if os.path.exists(test_cat) else 0
            print(f"\n   Kategori {cat}:")
            print(f"      TRAIN: {train_count} gambar")
            print(f"      TEST:  {test_count} gambar")

    # Simpan path ke variable environment supaya bisa dipakai di cell lain
    import os
    os.environ['TRAIN_PATH'] = train_path
    os.environ['TEST_PATH'] = test_path

else:
    print("\n Struktur folder yang ada:")
    !ls -la /content/data/

print(f"   train_path = '{train_path}'")
print(f"   test_path = '{test_path}'")

In [ ]:
import os

print("Mencari folder TRAIN dan TEST...")

# Cari dari root
for root, dirs, files in os.walk('/content'):
    for dir_name in dirs:
        if dir_name == 'TRAIN':
            train_path = os.path.join(root, dir_name)
            print(f"TRAIN ditemukan di: {train_path}")
        if dir_name == 'TEST':
            test_path = os.path.join(root, dir_name)
            print(f"TEST ditemukan di: {test_path}")

# Tampilkan struktur folder
print("\nStruktur folder /content:")
!ls -la /content/

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from collections import Counter

print("EDA - WASTE CLASSIFICATION DATASET")

train_path = '/content/data/DATASET/TRAIN'
test_path = '/content/data/DATASET/TEST'

classes = os.listdir(train_path)
print(f"Classes: {classes}")

train_counts = {}
test_counts = {}

for cls in classes:
    train_counts[cls] = len(os.listdir(os.path.join(train_path, cls)))
    test_counts[cls] = len(os.listdir(os.path.join(test_path, cls)))

print("\nDistribution:")
print(f"{'Class':<10} {'Train':<10} {'Test':<10} {'Total':<10}")
for cls in classes:
    print(f"{cls:<10} {train_counts[cls]:<10} {test_counts[cls]:<10} {train_counts[cls]+test_counts[cls]:<10}")

total_train = sum(train_counts.values())
total_test = sum(test_counts.values())

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(train_counts.keys(), train_counts.values(), color=['green', 'blue'])
axes[0].set_title('Training Set Distribution')
axes[0].set_ylabel('Number of Images')
axes[1].bar(test_counts.keys(), test_counts.values(), color=['green', 'blue'])
axes[1].set_title('Test Set Distribution')
axes[1].set_ylabel('Number of Images')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

# Pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie(train_counts.values(), labels=train_counts.keys(), autopct='%1.1f%%', colors=['green', 'blue'])
axes[0].set_title('Training Set')
axes[1].pie(test_counts.values(), labels=test_counts.keys(), autopct='%1.1f%%', colors=['green', 'blue'])
axes[1].set_title('Test Set')
plt.tight_layout()
plt.savefig('class_pie_chart.png', dpi=150)
plt.show()

# Sample images
print("\nSample Images:")
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, cls in enumerate(classes):
    cls_path = os.path.join(train_path, cls)
    samples = os.listdir(cls_path)[:4]
    for j, img_name in enumerate(samples):
        img_path = os.path.join(cls_path, img_name)
        img = load_img(img_path, target_size=(128, 128))
        img_array = img_to_array(img) / 255.0
        axes[idx, j].imshow(img_array)
        axes[idx, j].set_title(f"{cls}")
        axes[idx, j].axis('off')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150)
plt.show()

# Image resolution analysis
print("\nImage Resolution Analysis:")
resolutions = []
for cls in classes:
    cls_path = os.path.join(train_path, cls)
    samples = os.listdir(cls_path)[:50]
    for img_name in samples:
        img_path = os.path.join(cls_path, img_name)
        img = load_img(img_path)
        resolutions.append(img.size)
unique_resolutions = Counter(resolutions)
print(f"Unique resolutions: {len(unique_resolutions)}")
print(f"Most common: {unique_resolutions.most_common(3)}")

# Pixel intensity
print("\nPixel Intensity Distribution:")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for idx, cls in enumerate(classes):
    cls_path = os.path.join(train_path, cls)
    samples = os.listdir(cls_path)[:100]
    pixel_values = []
    for img_name in samples:
        img_path = os.path.join(cls_path, img_name)
        img = load_img(img_path, target_size=(128, 128))
        img_array = img_to_array(img) / 255.0
        pixel_values.extend(img_array.flatten())
    axes[idx].hist(pixel_values, bins=50, alpha=0.7, color=['green', 'blue'][idx])
    axes[idx].set_title(f'{cls} - Pixel Intensity')
    axes[idx].set_xlabel('Pixel Value')
    axes[idx].set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('pixel_intensity.png', dpi=150)
plt.show()

print("EDA SUMMARY")
print(f"Total images: {total_train + total_test}")
print(f"Classes: {classes}")
print(f"Train/Test split: {total_train}/{total_test}")
print(f"Class balance: Balanced")
print("EDA completed. All plots saved as PNG files.")